# Studio 2 — train di Colab

Latihannya lewat `train.py` di repo. Notebook cuma nyiapin GPU + dataset terus manggil skrip itu.

Set sebelum Run All: **Runtime > Change runtime type > T4 GPU**.

Sekarang isinya **Eksperimen 2**: nyoba nahan overfitting yang kejadian di Eksperimen 1
(val loss meledak di epoch 6-7). Yang diubah cuma 3 hal — lr diturunin, weight decay
dinaikin, augmentasi diperkuat. Bobot Eksperimen 1 di Drive nggak ditimpa, jadi bisa
dibandingin nanti.

## 1. Konfigurasi

Semua yang bisa berubah ada di sini. Ganti `EXP_NAME` kalau mau jalanin eksperimen lain —
nama itu yang dipakai buat nama bobot, nama grafik, dan nama riwayat, jadi tiap
eksperimen punya berkasnya sendiri dan nggak saling nimpa.

In [ ]:
# Repo yang di-clone di Colab.
REPO_URL = "https://github.com/SigitKurnia05/DL-RA_studio2-klasifikasi-sampah.git"
BRANCH = "main"

# Nama eksperimen: dipakai di nama bobot, grafik, dan riwayat.
EXP_NAME = "exp2_effnet_v2s_reg"

CANDIDATE = "submission/model.py"
OUT_WEIGHTS = f"/content/drive/MyDrive/Studio 2/Ckpt/{EXP_NAME}.pth"
PLOT_OUT = f"/content/drive/MyDrive/Studio 2/Ckpt/{EXP_NAME}_curves.png"
HIST_OUT = f"runs/{EXP_NAME}_history.json"

# Dataset: zip di Drive, diekstrak ke disk Colab (baca gambar dari Drive lambat).
DATASET_ZIP = "/content/drive/MyDrive/Studio 2/Dataset/train.zip"
DATA_DIR = "/content/train"

# Hyperparameter Eksperimen 2.
EPOCHS = 12
BATCH_SIZE = 32
LR = 1.5e-4     # turun dari 3e-4: lr kegedean bikin val loss meledak di exp 1
WD = 1e-2       # naik dari 1e-4: rem buat 21 juta parameter di 2.300 gambar
AUG = "strong"  # augmentasi kuat biar model nggak cepat hafal data latih
WORKERS = 2

print("repo   :", REPO_URL)
print("exp    :", EXP_NAME)
print("bobot  :", OUT_WEIGHTS)
print("grafik :", PLOT_OUT)
print("riwayat:", HIST_OUT)
print(f"latih  : {EPOCHS} epoch, bs {BATCH_SIZE}, lr {LR}, wd {WD}, aug {AUG}")

## 2. Nyiapin Colab

Mount Drive, clone repo, install paket tambahan, cek GPU-nya kerasa atau nggak.

Catatan: repo di-clone ulang dari GitHub, jadi `train.py` yang kepakai di sini adalah
versi yang sudah di-push (yang ada argumen `--aug`, `--wd`, `--history-out`).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

if not os.path.isdir("/content/repo/.git"):
    !git clone -b $BRANCH $REPO_URL /content/repo
%cd /content/repo
!git pull --ff-only

!pip install -q -r submission/requirements.txt

!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "TANPA GPU - set Runtime > Change runtime type > T4 GPU"
!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 3. Bongkar dataset

Zip diekstrak ke `DATA_DIR` (disk Colab), bukan dibaca langsung dari Drive.

Terus dicek jumlah gambar per kelas: harus 523 / 521 / 365 / 673 / 671. Kalau beda,
stop aja — datasetnya bukan yang dipakai waktu bikin `splits/val.txt`.

In [ ]:
import pathlib
import zipfile

CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]
EXPECT = {"botol_plastik": 523, "gelas_plastik": 521, "kertas_kardus": 365,
          "saset_kemasan": 673, "styrofoam": 671}


def cari_train(dasar):
    """Nyari folder yang isinya 5 subfolder kelas."""
    return [d for d in pathlib.Path(dasar).rglob("train")
            if d.is_dir() and all((d / c).is_dir() for c in CLASSES)]


zip_path = pathlib.Path(DATASET_ZIP)
root = pathlib.Path(DATA_DIR)
if not zip_path.is_file():
    raise SystemExit(f"[GAGAL] zip dataset tidak ada: {zip_path}")

if not root.exists():
    root.mkdir(parents=True)
    print("ekstrak", zip_path.name, "->", root)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(root)
else:
    print(root, "sudah ada, nggak diekstrak ulang")

ditemukan = cari_train(root)
if not ditemukan:
    raise SystemExit(f"[GAGAL] nggak ada folder train berisi 5 kelas di {root}")

DATA = max(ditemukan, key=lambda d: sum(len(list((d / c).glob("*.jpg"))) for c in CLASSES))
jumlah = {c: len(list((DATA / c).glob("*.jpg"))) for c in CLASSES}
print("dataset:", DATA, "| total", sum(jumlah.values()))
for c in CLASSES:
    print(f"  {c:16s} {jumlah[c]:5d}" + ("" if jumlah[c] == EXPECT[c] else f"   <-- harusnya {EXPECT[c]}"))
if any(jumlah[c] != EXPECT[c] for c in CLASSES):
    print("\n[PERINGATAN] snapshot dataset beda dari yang dipakai bikin splits/val.txt.")

## 4. Latih

Val-nya dibaca dari `splits/val.txt` (413 gambar), jadi angkanya sebanding sama Eksperimen 1.

Tiap epoch dicetak `train_loss`, `val_loss`, `macro_f1`, `acc`, `lr`, dan durasinya;
kalau macro_f1-nya rekor baru dikasih tanda `<-- REKOR`. Riwayatnya ditulis ke `HIST_OUT`.

Yang mau diliat di sini: **jarak train_loss vs val_loss**. Di Eksperimen 1 val loss naik
tajam mulai epoch 6 padahal train loss masih turun — itu tandanya hafal data latih.

In [ ]:
print(f"latih {EXP_NAME}: {EPOCHS} epoch, aug {AUG}")
!python train.py --model "{CANDIDATE}" --data "{DATA}" --out "{OUT_WEIGHTS}" --epochs {EPOCHS} --bs {BATCH_SIZE} --lr {LR} --wd {WD} --aug {AUG} --history-out "{HIST_OUT}"

## 5. Uji pakai evaluate.py

`evaluate.py` ini skrip yang sama persis dipakai asisten. Diuji di `/tmp/candidate`
biar `submission/` nggak kesentuh sebelum keputusan promosi.

Perhatiin `macro_f1` sama `acc`-nya, terus cek juga waktu inferensinya — batas dari asisten 15 menit.

In [ ]:
import pathlib, shutil

cand = pathlib.Path("/tmp/candidate")
cand.mkdir(exist_ok=True)
shutil.copy(CANDIDATE, cand / "model.py")
shutil.copy(OUT_WEIGHTS, cand / "model.pth")

!python evaluate.py --submission /tmp/candidate --data ./my_val --out /tmp/candidate/eval_{EXP_NAME}.json

## 6. Grafik

Baca `HIST_OUT`, terus gambar kurva loss, kurva macro-F1, dan confusion matrix dari hasil
evaluasi di atas. Disimpan ke `PLOT_OUT` di Drive.

Cara baca kurva loss: kalau val loss ikut turun bareng train loss, masih sehat. Kalau
val loss naik sementara train loss turun, di situ overfitting mulai.

In [ ]:
import json, pathlib

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.ticker import MaxNLocator

hist_path = pathlib.Path(HIST_OUT)
if not hist_path.is_file():
    raise SystemExit(f"[GAGAL] {hist_path} belum ada - jalanin sel latih dulu")

hist = json.loads(hist_path.read_text(encoding="utf-8"))
ep = [h["epoch"] for h in hist]
tr = [h["train_loss"] for h in hist]
va = [h["val_loss"] for h in hist]
f1 = [h["val_macro_f1"] for h in hist]
acc = [h["val_acc"] for h in hist]

puncak = max(range(len(f1)), key=lambda i: f1[i])
print(f"eksperimen        : {EXP_NAME}")
print(f"epoch tercatat    : {len(ep)}")
print(f"macro_f1 terbaik  : {f1[puncak]:.4f} (epoch {ep[puncak]})")
print(f"acc epoch itu     : {acc[puncak]:.4f}")
print(f"val loss terbaik  : {min(va):.4f} (epoch {ep[va.index(min(va))]})")
print(f"waktu total       : {sum(h['detik'] for h in hist):.0f} detik, "
      f"rata-rata {np.mean([h['detik'] for h in hist]):.1f} detik/epoch")

fig, ax = plt.subplots(1, 3, figsize=(19, 5.2))

ax[0].plot(ep, tr, "o-", color="#1f77b4", label="train")
ax[0].plot(ep, va, "s-", color="#d62728", label="val")
ax[0].set_title("Loss")
ax[0].set_xlabel("epoch")
ax[0].set_ylabel("cross-entropy")
ax[0].legend()
ax[0].grid(alpha=0.3)
ax[0].xaxis.set_major_locator(MaxNLocator(integer=True))

ax[1].plot(ep, f1, "o-", color="#2ca02c", label="macro_f1")
ax[1].plot(ep, acc, "^--", color="#7f7f7f", label="acc")
ax[1].axhline(0.92, color="#d62728", ls=":", lw=1.5, label="baseline asisten ~0.92")
ax[1].scatter([ep[puncak]], [f1[puncak]], s=180, marker="*", color="#ff7f0e", zorder=5,
              label=f"terbaik {f1[puncak]:.4f} (ep {ep[puncak]})")
ax[1].set_title("Macro-F1")
ax[1].set_xlabel("epoch")
ax[1].set_ylabel("macro_f1 / acc")
ax[1].legend(fontsize=9)
ax[1].grid(alpha=0.3)
ax[1].xaxis.set_major_locator(MaxNLocator(integer=True))

laporan = pathlib.Path(f"/tmp/candidate/eval_{EXP_NAME}.json")
if laporan.is_file():
    m = json.loads(laporan.read_text(encoding="utf-8"))
    cm = np.array(m["confusion_matrix"])
    label = list(m.get("per_class_f1", {})) or [f"kelas {i}" for i in range(len(cm))]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=label, yticklabels=label, ax=ax[2])
    ax[2].set_title(f"Confusion matrix (macro_f1 {m['macro_f1']:.4f}, acc {m['accuracy']:.4f})")
else:
    ax[2].text(0.5, 0.5, "berkas evaluasi belum ada", ha="center", va="center")
    ax[2].set_title("Confusion matrix")
ax[2].set_xlabel("prediksi")
ax[2].set_ylabel("asli")
plt.setp(ax[2].get_xticklabels(), rotation=30, ha="right")

# Judul dibikin defensif: kalau sel ini dijalankan sendiri (sesi Colab putus, sel
# konfigurasi belum jalan), nggak error — cuma keterangan setelannya kosong.
setelan = ", ".join(f"{k} {v}" for k, v in (("lr", globals().get("LR")), ("wd", globals().get("WD")),
                                            ("aug", globals().get("AUG"))) if v is not None)
fig.suptitle(globals().get("EXP_NAME", "eksperimen") + (f" — {setelan}" if setelan else ""))
fig.tight_layout()

keluar = pathlib.Path(PLOT_OUT)
keluar.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(keluar, dpi=150, bbox_inches="tight")
print("grafik disimpan:", keluar)
plt.show()

## 7. Cek bobotnya

Ukurannya dibatasin GitHub 100 MB per file (di atas itu mesti lewat GitHub Release atau LFS).
Perhatiin juga: bobot Eksperimen 1 nggak ditimpa karena namanya beda.

In [ ]:
import hashlib, json, pathlib

bobot = pathlib.Path(OUT_WEIGHTS)
mb = bobot.stat().st_size / 2**20
print(f"{bobot}: {mb:.1f} MB (batas GitHub 100 MB)")
print("sha256:", hashlib.sha256(bobot.read_bytes()).hexdigest())
if mb > 100:
    print("[CATATAN] lewat 100 MB: pakai GitHub Release atau LFS, link-nya tulis di README")

laporan = pathlib.Path(f"/tmp/candidate/eval_{EXP_NAME}.json")
if laporan.is_file():
    m = json.loads(laporan.read_text(encoding="utf-8"))
    print(f"macro_f1 {m['macro_f1']:.4f} | acc {m['accuracy']:.4f} | {m['n_images']} gambar | {m['seconds']} detik")

from google.colab import files

files.download(OUT_WEIGHTS)
files.download(PLOT_OUT)

## 8. Habis ini ngapain

Bobotnya udah aman di Drive dengan nama `EXP_NAME`, jadi Eksperimen 1 tetap utuh buat
pembanding.

Bandingin dua hal sebelum promosi ke `submission/`:
- **macro_f1** — mana yang lebih tinggi di val yang sama (413 gambar).
- **bentuk kurva** — kalau val loss-nya naik lagi di akhir, berarti overfitting-nya masih ada
  walau angkanya kebetulan lebih tinggi.

Kalau Eksperimen 2 menang, promosi = satu commit buat dua file (`submission/model.py` +
`submission/model.pth`); kalau salah satu ketinggalan, `evaluate.py` gagal load (`strict=True`).